# Headless Dafne Thigh segmentation — Water images (Lambda)

Runs the Dafne Thigh model on all WATER NIfTI stacks found under `~/myosegmenTUM/`.  
No GUI required — uses `dafne_dl.DynamicDLModel` directly.

## Before running

Install dependencies (once per instance):
```bash
pip install dafne-dl SimpleITK
```

**Upload model:**
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/dafne_thigh_results/model_used/ \
  ubuntu@129.80.59.179:~/dafne_model/
```

## Download results when done
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@129.80.59.179:~/dafne_water_segs/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/dafne_water_segs/
```

**Terminate the instance when done.**

In [ ]:
import glob
import os
import numpy as np
import SimpleITK as sitk
from dafne_dl import DynamicDLModel

In [ ]:
# --- paths ---
MODEL_PATH = os.path.expanduser('~/dafne_model/Thigh_1774532147.model')
IMAGE_GLOB = os.path.expanduser('~/myosegmenTUM/*/ImageData/*_WATER/*_WATER_stack*.nii')
OUTPUT_DIR = os.path.expanduser('~/dafne_water_segs')

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# load model once — DynamicDLModel.Load reads the local .model file
model = DynamicDLModel.Load(open(MODEL_PATH, 'rb'))
print('Model loaded:', MODEL_PATH)

In [ ]:
image_files = sorted(glob.glob(IMAGE_GLOB))
print(f'Found {len(image_files)} images:')
for p in image_files:
    print(' ', p)

In [ ]:
# run segmentation slice-by-slice and save one .npz per stack
for nii_path in image_files:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUTPUT_DIR, f'{stem}_dafne_thigh.npz')

    if os.path.exists(out_path):
        print(f'Skipping (already done): {out_path}')
        continue

    print(f'\nProcessing: {nii_path}')

    img_sitk   = sitk.ReadImage(nii_path)
    img_array  = sitk.GetArrayFromImage(img_sitk).astype(float)  # (slices, H, W)
    spacing    = img_sitk.GetSpacing()                            # (x_mm, y_mm, z_mm)
    resolution = [spacing[0], spacing[1]]                         # 2D in-plane spacing

    print(f'  Shape: {img_array.shape}  Resolution: {resolution}')

    all_masks = {}  # {muscle_name: 3D uint8 array (slices, H, W)}

    for slice_idx in range(img_array.shape[0]):
        slice_2d = img_array[slice_idx]
        out = model({
            'image':          slice_2d,
            'resolution':     resolution,
            'split_laterality': True,
            'classification': 'Thigh',
        })

        for muscle_name, mask in out.items():
            if muscle_name not in all_masks:
                all_masks[muscle_name] = np.zeros(img_array.shape, dtype=np.uint8)
            all_masks[muscle_name][slice_idx] = np.asarray(mask, dtype=np.uint8)

        if (slice_idx + 1) % 5 == 0 or slice_idx == img_array.shape[0] - 1:
            print(f'  slice {slice_idx + 1}/{img_array.shape[0]} done')

    np.savez_compressed(out_path, **all_masks)
    print(f'  Saved → {out_path}')
    print(f'  Muscles: {list(all_masks.keys())}')

print('\nAll done.')

In [ ]:
# quick sanity check — reload one result and show muscle names + voxel counts
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.npz')))
if results:
    sample = np.load(results[0])
    print('Sample file:', results[0])
    for name in sample.files:
        arr = sample[name]
        print(f'  {name}: shape={arr.shape}  positive voxels={arr.sum()}')